In [9]:
"""
Day 003 - Adaline: full-batch GD vs SGD, and why scaling decides both.

The perceptron with the threshold moved. The loss is measured on the raw net input, before any step function, so there's a real gradient to descend.

"""

import numpy as np
from sklearn.datasets import load_iris

class AdalineGD:

    def __init__(self, lr=0.01, epochs=100, seed=1):
        self.lr, self.epochs, self.seed = lr, epochs, seed

    def net_input(self, X):
        return X @ self.w_ + self.b_

    def activation(self, z):
        return z

    def predict(self, X):
        return np.where(self.activation(self.net_input(X)) >= 0.0, 1, -1)

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        self.w_ = rng.normal(scale=0.01, size=X.shape[1])
        self.b_, self.losses_ = 0.0, []
        n = X.shape[0]
        for _ in range(self.epochs):
            resid = self.activation(self.net_input(X)) - y
            self.losses_.append(np.mean(resid ** 2))
            self.w_ -= self.lr * (2/n) * (X.T @ resid)
            self.b_ -= self.lr * (2/n) * resid.sum()
        return self

class AdalineSGD(AdalineGD):

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        self.w_ = rng.normal(scale=0.01, size=X.shape[1])
        self.b_, self.losses_ = 0.0, []
        for _ in range(self.epochs):
            epoch_loss = []
            for i in rng.permutation(X.shape[0]):
                resid = (X[i] @ self.w_ + self.b_) -y[i]
                epoch_loss.append(resid ** 2)
                self.w_ -= self.lr * 2.0 * resid * X[i]
                self.b_ -= self.lr * 2.0 * resid
            self.losses_.append(np.mean(epoch_loss))
        return self

    def partial_fit(self, xi, target):
        """Online learning: the whole point of SGD. No refit, no stored data."""
        resid = (xi @ self.w_ + self.b_) - target
        self.w_ -= self.lr * 2.0 * resid * xi
        self.b_ -= self.lr * 2.0 * resid
        return self

def iris_two_class():
    """Setosa vs. versicolor, sepal length and petal length, labels -1/+1"""
    d = load_iris()
    mask = d.target < 2
    X = d.data[mask][:, [0,2]]
    y = np.where(d.target[mask] == 0, -1.0, 1.0)
    return X, y

def closed_form(X, y):
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    theta = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
    return theta[1:], theta[0]

if __name__ == "__main__":
    X, y = iris_two_class()
    Xs = (X - X.mean(axis=0)) / X.std(axis=0)

    # 1 raw features: one lr explodes, a survivable one barely moves
    with np.errstate(over='ignore', invalid='ignore'):
        big = AdalineGD(lr=0.1, epochs=15).fit(X,y)
    small = AdalineGD(lr=0.0001, epochs=15).fit(X,y)
    print("raw, lr=0.1 MSE:", np.round(big.losses_[:4], 3), "->", big.losses_[-1])
    print("raw, lr=0.0001 MSE", np.round(small.losses_[:4], 4), "->", round(small.losses_[-1], 4))

    #. same data, standardized: a large lr is now safe
    gd = AdalineGD(lr=0.5, epochs=20).fit(Xs, y)
    sgd = AdalineSGD(lr=0.01, epochs=20).fit(Xs, y)
    print("scaled GD MSE per epoch:", np.round(gd.losses_[:6], 4), "->", round(gd.losses_[-1], 4))
    print("scaled SGD MSE per epoch:", np.round(sgd.losses_[:6], 4), "->", round(sgd.losses_[-1], 4))
    print("accuracy GD / SGD:", np.mean(gd.predict(Xs) == y),
          np.mean(sgd.predict(Xs) == y))

    # 3. online update the same bowl Day 001 solved in one shot
    sgd.partial_fit(Xs[0], y[0])

    w_cf, b_cf = closed_form(Xs, y)
    long_gd = AdalineGD(lr=0.5, epochs=500).fit(Xs, y)
    print("closed form :", np.round(np.append(w_cf, b_cf), 4))
    print("adaline @500ep  :", np.round(np.append(long_gd.w_, long_gd.b_), 4))
    assert np.allclose(long_gd.w_, w_cf, atol=1e-4)
    assert np.isclose(long_gd.b_, b_cf, atol=1e-4)
    print("adaline converges to the normal equation")


raw, lr=0.1 MSE: [9.760000e-01 1.389000e+00 3.999100e+01 1.996181e+03] -> 1.0629014726413486e+22
raw, lr=0.0001 MSE [0.9758 0.9751 0.9745 0.9738] -> 0.967
scaled GD MSE per epoch: [0.9792 0.6628 0.4541 0.3162 0.2253 0.1652] -> 0.049
scaled SGD MSE per epoch: [0.2762 0.1045 0.0754 0.0624 0.056  0.0531] -> 0.0493
accuracy GD / SGD: 1.0 1.0
closed form : [-0.1759  1.1129 -0.    ]
adaline @500ep  : [-0.1759  1.1129 -0.    ]
adaline converges to the normal equation
